In [3]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict

In [4]:
load_dotenv()

True

In [19]:
model = ChatGroq(model='openai/gpt-oss-120b')

In [5]:
class LLM(TypedDict):
    question:str
    answer:str

In [21]:
def q_a(state:LLM) -> LLM :
    question = state['question']
    prompt = f' answer the following the {question} from your knowledge base if do not know tell clearly '
    result = model.invoke(prompt)
    state['answer'] = result.content
    return state

In [22]:
graph = StateGraph(LLM)
graph.add_node('llm_q_a',q_a)
graph.add_edge(START,'llm_q_a')
graph.add_edge('llm_q_a',END)

workflow = graph.compile()

In [25]:
#execute

inital_q = {'question':'which model are you and what is the hottest planet in solar system'}
final_state = workflow.invoke(inital_q)
print(final_state['answer'])

I am Qwen, a large language model independently developed by Alibaba Group's Tongyi Lab.

Based on my knowledge, the hottest planet in the solar system is **Venus**. Despite Mercury being closer to the Sun, Venus has a much thicker atmosphere composed mainly of carbon dioxide, which creates a strong greenhouse effect, trapping heat and raising surface temperatures to around 465°C (870°F).


# Prompt Chaining 

In [11]:
class Outliner(TypedDict):
    topic:str
    outline:str
    blog:str

In [7]:
def prompter(state:Outliner)-> Outliner:
    topic = state['topic']
    prompt = f" You are a blog writer Make a outline for this {topic}"
    result =model.invoke(prompt)
    state['outline'] = result.content
    return state

In [23]:
def blogger(state:Outliner) -> Outliner:
    outline = state['outline']
    prompt = f'this is the {outline} of a blog make a clean proffesional blog out of it and make sure to follow the outline given in 150 words'
    blog = model.invoke(prompt)
    state['blog'] = blog.content
    return state

In [24]:
graph = StateGraph(Outliner)
graph.add_node('outline',prompter)
graph.add_node('blog',blogger)
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog',END)
workflow=graph.compile()

In [25]:
topic={'topic':'Machine learning'}
final_output = workflow.invoke(topic)
print(final_output['outline'])

**Blog Title:** *Demystifying Machine Learning: A Beginner‑Friendly Guide to How It Works, Why It Matters, and Where It’s Heading*

---

## 1️⃣ Introduction  
- **Hook:** A compelling anecdote or statistic (e.g., “Every 60 seconds, Google processes 40,000 searches powered by ML”).  
- **Definition:** One‑sentence, jargon‑free definition of machine learning (ML).  
- **Purpose of the Post:** What readers will learn and why it matters to them (career, daily life, future tech).  
- **Brief Roadmap:** Quick preview of the sections to come.

---

## 2️⃣ The Foundations of Machine Learning  
### 2.1. What Is “Learning” in a Computer?  
- Supervised vs. unsupervised vs. reinforcement learning – intuitive analogies.  
### 2.2. Core Components  
- **Data:** Types (structured, unstructured, streaming).  
- **Model:** What a model is (function that maps inputs → outputs).  
- **Loss / Objective Function:** How we measure “mistakes.”  
- **Optimization:** Gradient descent explained with a visual m